In [70]:
!pip install monai
!pip install argparse


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [83]:
import monai
from monai.data import PILReader
from monai.transforms import (
    # AsChannelFirstd,
    # AddChanneld,
    Compose,
    LoadImaged,
    SpatialPadd,
    RandSpatialCropd,
    RandRotate90d,
    ScaleIntensityd,
    RandAxisFlipd,
    RandZoomd,
    RandGaussianNoised,
    RandAdjustContrastd,
    RandGaussianSmoothd,
    RandHistogramShiftd,
    EnsureTyped,
)
from torch.utils.data import DataLoader
import torch
import numpy as np
import os

In [72]:
class Args:
    def __init__(self):
        self.data_path = 'code\\baseline\\data\\Train_Labeled'
        self.input_size = 256

    def set_input_size(self, size):
        self.input_size = size

    def set_data_path(self, path):
        self.data_path = path

In [73]:
join = os.path.join
args = Args()

In [74]:
train_transforms = Compose(
        [
            LoadImaged(
                keys=["img", "label"], reader=PILReader, dtype=np.uint8
            ),  # image three channels (H, W, 3); label: (H, W)
            # AddChanneld(keys=["label"], allow_missing_keys=True),  # label: (1, H, W)
            # AsChannelFirstd(
            #     keys=["img"], channel_dim=-1, allow_missing_keys=True
            # ),  # image: (3, H, W)
            ScaleIntensityd(
                keys=["img"], allow_missing_keys=True
            ),  # Do not scale label
            SpatialPadd(keys=["img", "label"], spatial_size=args.input_size),
            RandSpatialCropd(
                keys=["img", "label"], roi_size=args.input_size, random_size=False
            ),
            RandAxisFlipd(keys=["img", "label"], prob=0.5),
            RandRotate90d(keys=["img", "label"], prob=0.5, spatial_axes=[0, 1]),
            # # intensity transform
            RandGaussianNoised(keys=["img"], prob=0.25, mean=0, std=0.1),
            RandAdjustContrastd(keys=["img"], prob=0.25, gamma=(1, 2)),
            RandGaussianSmoothd(keys=["img"], prob=0.25, sigma_x=(1, 2)),
            RandHistogramShiftd(keys=["img"], prob=0.25, num_control_points=3),
            RandZoomd(
                keys=["img", "label"],
                prob=0.15,
                min_zoom=0.8,
                max_zoom=1.5,
                mode=["area", "nearest"],
            ),
            EnsureTyped(keys=["img", "label"]),
        ]
    )

In [78]:
val_transforms = Compose(
    [
        LoadImaged(keys=["img", "label"], reader=PILReader, dtype=np.uint8),
        # AddChanneld(keys=["label"], allow_missing_keys=True),
        # AsChannelFirstd(keys=["img"], channel_dim=-1, allow_missing_keys=True),
        ScaleIntensityd(keys=["img"], allow_missing_keys=True),
        # AsDiscreted(keys=['label'], to_onehot=3),
        EnsureTyped(keys=["img", "label"]),
    ]
)

In [75]:
img_path = join(args.data_path, "images")
gt_path = join(args.data_path, "labels")

In [ ]:
img_names = sorted(os.listdir(img_path))
gt_names = [img_name.split(".")[0] + "_label.png" for img_name in img_names]
img_num = len(img_names)
val_frac = 0.1
indices = np.arange(img_num)
np.random.shuffle(indices)
val_split = int(img_num * val_frac)
train_indices = indices[val_split:]
val_indices = indices[:val_split]

In [ ]:
train_files = [
    {"img": join(img_path, img_names[i]), "label": join(gt_path, gt_names[i])}
    for i in train_indices
]

In [ ]:
val_files = [
    {"img": join(img_path, img_names[i]), "label": join(gt_path, gt_names[i])}
    for i in val_indices
]

In [84]:
check_ds = monai.data.Dataset(data=train_files, transform=train_transforms)
check_loader = DataLoader(check_ds, batch_size=1, num_workers=4)
check_data = monai.utils.misc.first(check_loader)
print(
    "sanity check:",
    check_data["img"].shape,
    torch.max(check_data["img"]),
    check_data["label"].shape,
    torch.max(check_data["label"]),
)

RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\Samir\AppData\Local\Programs\Python\Python311\Lib\site-packages\monai\transforms\transform.py", line 141, in apply_transform
    return _apply_transform(transform, data, unpack_items, lazy, overrides, log_stats)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Samir\AppData\Local\Programs\Python\Python311\Lib\site-packages\monai\transforms\transform.py", line 98, in _apply_transform
    return transform(data, lazy=lazy) if isinstance(transform, LazyTrait) else transform(data)
                                                                               ^^^^^^^^^^^^^^^
  File "c:\Users\Samir\AppData\Local\Programs\Python\Python311\Lib\site-packages\monai\transforms\io\dictionary.py", line 163, in __call__
    data = self._loader(d[key], reader)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Samir\